In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:

import glob
import os
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import re



### Visualizing `mean_HU`, `min_HU`, and `max_HU` Comparison Across All `CEJ` Folders

In [ ]:


# Định nghĩa thư mục gốc và các thư mục con cần so sánh
base_dir = '/content/drive/MyDrive/CEJ/P2'
folders = ['1', '2', '3', '4', '5']

# Thu thập toàn bộ các file từ các thư mục
all_filenames_across_folders = set()
file_paths_by_folder = {folder: {} for folder in folders}

for folder in folders:
    current_path = os.path.join(base_dir, folder, '*.csv')
    for f_path in glob.glob(current_path):
        filename = os.path.basename(f_path)
        all_filenames_across_folders.add(filename)
        file_paths_by_folder[folder][filename] = f_path

unique_filenames = sorted(list(all_filenames_across_folders))

# TỐI ƯU HÓA: Đọc file MỘT LẦN và vẽ cả 3 biểu đồ (mean_HU, min_HU, max_HU)
for filename in unique_filenames:
    # Tạo 3 subplots cho mean, min, max
    fig, axes = plt.subplots(1, 3, figsize=(18, 5), sharex=True)
    has_data = False

    for folder in folders:
        if filename in file_paths_by_folder[folder]:
            file_path = file_paths_by_folder[folder][filename]
            try:
                df = pd.read_csv(file_path)

                # Vẽ mean_HU
                if 'mean_HU' in df.columns:
                    sns.lineplot(x=df.index, y=df['mean_HU'], ax=axes[0], label=f'CEJ/{folder}')
                    has_data = True

                # Vẽ min_HU
                if 'min_HU' in df.columns:
                    sns.lineplot(x=df.index, y=df['min_HU'], ax=axes[1], label=f'CEJ/{folder}', errorbar=None)

                # Vẽ max_HU
                if 'max_HU' in df.columns:
                    sns.lineplot(x=df.index, y=df['max_HU'], ax=axes[2], label=f'CEJ/{folder}', errorbar=None)

            except Exception as e:
                print(f"Lỗi khi đọc {file_path}: {e}")

    if has_data:
        axes[0].set_title(f'mean_HU: {filename}')
        axes[1].set_title(f'min_HU: {filename}')
        axes[2].set_title(f'max_HU: {filename}')

        for ax in axes:
            ax.set_xlabel('Slice Index')
            ax.set_ylabel('Giá trị HU')
            ax.legend(title='Folder')
            ax.grid(True, linestyle='--', alpha=0.7)

        plt.tight_layout()
        plt.show()
    else:
        plt.close(fig)
        print(f"Không tìm thấy dữ liệu HU cho {filename}.")

### Đề xuất phân đoạn lát cắt thành Men răng (Enamel) và Xi măng (Cementum) dựa trên giá trị HU

In [ ]:

# Define the path for CEJ/3
cej3_path = '/content/drive/MyDrive/CEJ/P2/3/*.csv'
csv_files_cej3 = glob.glob(cej3_path)

cej3_dataframes = {}
for file_path in csv_files_cej3:
    filename = os.path.basename(file_path)
    try:
        df = pd.read_csv(file_path)
        cej3_dataframes[filename] = df
    except Exception as e:
        print(f"Lỗi khi đọc tệp {file_path}: {e}")
        continue

def safe_mean(arr):
    if len(arr) == 0: return np.nan
    return np.mean(arr)

def safe_std(arr):
    if len(arr) < 2: return 0.0
    return np.std(arr)

print("Phân tích 'mean_HU', 'std_HU' và 'max_HU' để phân định 3 vùng (chỉ nằm trong khoảng 10%-90%):\n")

# Dictionary to store calculation results for reuse in later cells
cej3_results = {}

for filename, df in cej3_dataframes.items():
    if not all(col in df.columns for col in ['mean_HU', 'std_HU', 'max_HU']):
        print(f"Tệp {filename} thiếu cột cần thiết. Bỏ qua.\n")
        continue

    mean_hu = df['mean_HU'].values
    std_hu = df['std_HU'].values
    max_hu = df['max_HU'].values
    n_slices = len(mean_hu)

    calc_start = int(np.floor(0.10 * n_slices))
    calc_end = int(np.ceil(0.90 * n_slices))
    valid_len = calc_end - calc_start

    min_stable_len = max(2, int(np.floor(0.20 * valid_len))) # Vùng ổn định dài ít nhất 20%

    # Xác định tỷ lệ vùng chuyển tiếp theo loại răng
    if re.search(r'fdi[1-4]1', filename):
        trans_ratio = 0.25
    elif re.search(r'fdi[1-4][34]', filename):
        trans_ratio = 0.20
    elif re.search(r'fdi[1-4]6', filename):
        trans_ratio = 0.15
    else:
        trans_ratio = 0.20

    max_trans_len = max(4, int(np.floor(trans_ratio * valid_len)))
    min_trans_len = 2

    best_i = -1
    best_j = -1
    max_score = -np.inf

    # Iterate over all possible combinations of (i, j) STRICTLY WITHIN [calc_start, calc_end]
    for i in range(calc_start + min_stable_len, calc_end - min_stable_len - min_trans_len + 1):
        for j in range(i + min_trans_len, min(i + max_trans_len + 1, calc_end - min_stable_len + 1)):
            # Calculate means (strictly restricted to zones inside 10%-90%)
            m1_mean, m3_mean = safe_mean(mean_hu[calc_start:i]), safe_mean(mean_hu[j:calc_end])
            m1_max, m3_max = safe_mean(max_hu[calc_start:i]), safe_mean(max_hu[j:calc_end])

            # Calculate stdevs (internal variance of the zones, strictly restricted)
            std1_mean, std3_mean = safe_std(mean_hu[calc_start:i]), safe_std(mean_hu[j:calc_end])
            std1_max, std3_max = safe_std(max_hu[calc_start:i]), safe_std(max_hu[j:calc_end])
            std1_stdhu, std3_stdhu = safe_std(std_hu[calc_start:i]), safe_std(std_hu[j:calc_end])

            # Độ chênh lệch giữa 2 vùng ổn định
            drop_score = abs(m1_mean - m3_mean) + abs(m1_max - m3_max)

            # Đưa vùng chuyển tiếp về gần phía thân răng (vùng có HU lớn)
            if m1_mean > m3_mean:
                distance_ratio = (i - calc_start) / valid_len
            else:
                distance_ratio = (calc_end - j) / valid_len

            position_weight = 1.0 - 0.5 * distance_ratio
            penalty = (std1_mean + std3_mean) + (std1_max + std3_max) + (std1_stdhu + std3_stdhu)
            score = (drop_score * position_weight) / (penalty + 1e-5)

            if score > max_score:
                max_score = score
                best_i = i
                best_j = j

    fig, ax = plt.subplots(figsize=(12, 6))

    sns.lineplot(x=df.index, y=df['mean_HU'], ax=ax, label='Mean HU', color='purple')
    sns.lineplot(x=df.index, y=df['max_HU'], ax=ax, label='Max HU', color='lime', linestyle=':')
    sns.lineplot(x=df.index, y=df['std_HU'], ax=ax, label='Std HU', color='cyan', linestyle='--')

    if best_i != -1 and best_j != -1:
        ax.axvline(x=best_i, color='red', linestyle='-', linewidth=2, label=f'Điểm chia 1: {best_i}')
        ax.axvline(x=best_j, color='blue', linestyle='-', linewidth=2, label=f'Điểm chia 2: {best_j}')

        ax.axvspan(calc_start, best_i, alpha=0.1, color='green', label='Vùng 1: Ổn định (1)')
        ax.axvspan(best_i, best_j, alpha=0.1, color='orange', label=f'Vùng 2: Chuyển tiếp (Tối đa {int(trans_ratio*100)}%)')
        ax.axvspan(best_j, calc_end, alpha=0.1, color='blue', label='Vùng 3: Ổn định (2)')

        ax.axvspan(0, calc_start, alpha=0.3, color='gray', label='Bỏ qua (10% đầu)')
        ax.axvspan(calc_end, n_slices, alpha=0.3, color='gray', label='Bỏ qua (10% cuối)')

    ax.set_title(f'Phân định 3 vùng HU cho tệp: {filename}')
    ax.set_xlabel('Chỉ số lát cắt (Slice Index)')
    ax.set_ylabel('Giá trị HU')
    ax.legend()
    ax.grid(True, linestyle='--', alpha=0.7)
    plt.tight_layout()
    plt.show()

    print(f"Cho tệp '{filename}':")
    if best_i != -1 and best_j != -1:
        direction = "Giảm dần" if safe_mean(mean_hu[calc_start:best_i]) > safe_mean(mean_hu[best_j:calc_end]) else "Tăng dần"

        # Save calculation for next cells
        cej3_results[filename] = {
            'best_i': best_i,
            'best_j': best_j,
            'direction': direction,
            'trans_ratio': trans_ratio
        }

        print(f"- Vùng 1 (Ổn định 1): Lát cắt {calc_start} đến {best_i}")
        print(f"  Mean HU: {safe_mean(mean_hu[calc_start:best_i]):.2f}, Max HU: {safe_mean(max_hu[calc_start:best_i]):.2f}, Std HU: {safe_mean(std_hu[calc_start:best_i]):.2f}")
        print(f"- Vùng 2 (Chuyển tiếp - {direction} - Ngắn): Lát cắt {best_i} đến {best_j}")
        print(f"  Mean HU: {safe_mean(mean_hu[best_i:best_j]):.2f}, Max HU: {safe_mean(max_hu[best_i:best_j]):.2f}, Std HU: {safe_mean(std_hu[best_i:best_j]):.2f}")
        print(f"- Vùng 3 (Ổn định 2): Lát cắt {best_j} đến {calc_end}")
        print(f"  Mean HU: {safe_mean(mean_hu[best_j:calc_end]):.2f}, Max HU: {safe_mean(max_hu[best_j:calc_end]):.2f}, Std HU: {safe_mean(std_hu[best_j:calc_end]):.2f}")
    else:
        cej3_results[filename] = None
        print("- Không tìm thấy 3 vùng thỏa mãn điều kiện.")
    print("\n---\n")

In [ ]:


ratio_data = []

for filename, df in cej3_dataframes.items():
    if not all(col in df.columns for col in ['z_mm']):
        continue

    # Tái sử dụng kết quả từ cell trước đó để tránh tính toán lại
    result = cej3_results.get(filename)
    if not result:
        continue

    best_i = result['best_i']
    best_j = result['best_j']
    direction = result['direction']
    trans_ratio = result['trans_ratio']
    n_slices = len(df)

    # Phân định độ dài thân và chân tùy vào hướng
    if direction == "Giảm dần":
        crown_trans_slices = best_j
        root_slices = n_slices - best_j
        crown_trans_z = abs(df['z_mm'].iloc[best_j-1] - df['z_mm'].iloc[0])
        root_z = abs(df['z_mm'].iloc[-1] - df['z_mm'].iloc[best_j]) if root_slices > 0 else 0
    else:
        crown_trans_slices = n_slices - best_i
        root_slices = best_i
        crown_trans_z = abs(df['z_mm'].iloc[-1] - df['z_mm'].iloc[best_i])
        root_z = abs(df['z_mm'].iloc[best_i-1] - df['z_mm'].iloc[0]) if root_slices > 0 else 0

    ratio_slices = crown_trans_slices / root_slices if root_slices > 0 else np.nan
    ratio_z_mm = crown_trans_z / root_z if root_z > 0 else np.nan

    ratio_data.append({
        'filename': filename,
        'direction': direction,
        'crown_trans_slices': crown_trans_slices,
        'root_slices': root_slices,
        'crown_root_ratio_slices': round(ratio_slices, 3),
        'crown_trans_mm': round(crown_trans_z, 2),
        'root_mm': round(root_z, 2),
        'crown_root_ratio_mm': round(ratio_z_mm, 3),
        'trans_ratio_used': trans_ratio
    })

crown_root_ratio_df = pd.DataFrame(ratio_data)
display(crown_root_ratio_df)

In [ ]:
# Xuất bảng tỷ lệ Thân/Chân răng ra file CSV để lưu trữ
output_csv_path = '/content/drive/MyDrive/CEJ/P2/crown_root_ratio_dynamic_10_90.csv'

# Lưu trữ với mã hóa utf-8-sig để hỗ trợ tiếng Việt tốt nếu có
crown_root_ratio_df.to_csv(output_csv_path, index=False, encoding='utf-8-sig')

print(f"Đã xuất dữ liệu thành công ra file: {output_csv_path}")
